# 資料處理

## 讀nc4

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm


def check_data_folder(folder: str) -> bool:
    return os.path.exists(folder) and os.path.isdir(folder)


def load_data(file_path: str) -> xr.Dataset:
    """
    Load data from a NetCDF file, trying netcdf4 then h5netcdf.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    # 優先 netcdf4
    try:
        return xr.open_dataset(file_path, engine="netcdf4")
    except Exception as e1:
        # 改試 h5netcdf
        try:
            return xr.open_dataset(file_path, engine="h5netcdf")
        except Exception as e2:
            raise RuntimeError(
                f"Failed to open {file_path} with netcdf4 and h5netcdf.\n"
                f"e1: {e1}\n"
                f"e2: {e2}\n"
                "請確認這個環境有安裝 netCDF4 或 h5netcdf。"
            )


# ================= main program =================

data_folder = "nc4"
var_name = "T2M"  # 目標變數名稱（請確認檔內真的叫這個）

# 1. 檢查資料夾
if not check_data_folder(data_folder):
    raise FileNotFoundError(f"Data folder not found: {data_folder}")
print(f"Data folder found: {data_folder}")

# 2. 找出所有像 1980-01.nc4 的檔案
pattern = os.path.join(data_folder, "*.nc4")
file_list = sorted(glob.glob(pattern))

if len(file_list) == 0:
    raise FileNotFoundError(f"No nc4 files found with pattern: {pattern}")

print(f"Found {len(file_list)} files.")
print("First 5 files:", file_list[:5])

# 3. 用第一個檔案確認經緯度與變數存在，必要時自動偵測 var_name
with load_data(file_list[0]) as sample_data:
    print("Data variables in first file:")
    print(list(sample_data.data_vars))
    print("Coordinates:")
    print({k: sample_data[k].shape for k in sample_data.coords})

    # 找出所有長得像 (time, lat, lon) 的候選變數
    candidates = []
    for v in sample_data.data_vars:
        dims = set(sample_data[v].dims)
        if {"time", "lat", "lon"}.issubset(dims):
            candidates.append(v)

    # 如果原本設定的 var_name 不在，就試著自動改
    if var_name not in sample_data.data_vars:
        if len(candidates) == 1:
            auto_var = candidates[0]
            print(f"[Info] Variable '{var_name}' not found, auto-select '{auto_var}' as target.")
            var_name = auto_var
        else:
            raise KeyError(
                f"Variable '{var_name}' not found in file: {file_list[0]}\n"
                f"Available data_vars: {list(sample_data.data_vars)}\n"
                f"3D (time,lat,lon) candidates: {candidates}\n"
                "請將上方其中一個正確變數名稱填入 var_name。"
            )

    # 取經緯度
    if "lat" not in sample_data.coords or "lon" not in sample_data.coords:
        raise KeyError("lat/lon coordinates not found in sample file.")
    lat = sample_data["lat"].values
    lon = sample_data["lon"].values

nlat = lat.shape[0]
nlon = lon.shape[0]
print(f"Confirmed var_name = {var_name}")
print(f"Lat: {nlat}, Lon: {nlon}")


# 4. 逐檔讀入，累積到 list
data_list = []
time_list = []

for f in tqdm(file_list, desc="Combining"):
    with load_data(f) as ds:
        da = ds[var_name]  # (time, lat, lon)

        # 確保 lat/lon 一致（保險，可視情況註解）
        if da.sizes["lat"] != nlat or da.sizes["lon"] != nlon:
            raise ValueError(f"Lat/Lon size mismatch in file: {f}")

        # 資料轉 float32，省記憶體
        data_list.append(da.values.astype(np.float32))

        if "time" not in ds:
            raise KeyError(f"'time' coordinate not found in file: {f}")

        # decode_cf 確保時間是真正 datetime
        t = xr.decode_cf(ds)["time"].values
        time_list.append(t.astype("datetime64[ns]"))

# 5. 串起來 → (ntot, lat, lon) & DatetimeIndex
combined = np.concatenate(data_list, axis=0)   # (ntot, nlat, nlon)
time_array = np.concatenate(time_list, axis=0) # (ntot,)

if combined.shape[0] != time_array.shape[0]:
    raise ValueError(
        f"time length ({time_array.shape[0]}) "
        f"!= data length ({combined.shape[0]})"
    )

# 依時間排序（通常已排序，這裡是保險）
sort_idx = np.argsort(time_array)
combined = combined[sort_idx]
time_array = time_array[sort_idx]

time_index = pd.to_datetime(time_array)

print(f"Combined data shape: {combined.shape}")
print(f"Time index: {time_index[0]} -> {time_index[-1]} (len={len(time_index)})")

# 6. 攤平成 cell × time
ntot, nlat, nlon = combined.shape
ncell = nlat * nlon

# (cell, time)
y_all = combined.reshape(ntot, ncell).T

# 建立每個 cell 的 (lon, lat)
lon_grid, lat_grid = np.meshgrid(lon, lat)
gg = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])  # (cell, 2)

print(f"y_all shape: {y_all.shape}  (cells x time)")
print(f"gg shape: {gg.shape}        (cells x [lon, lat])")


Data folder found: nc4
Found 548 files.
First 5 files: ['nc4/1980-01.nc4', 'nc4/1980-02.nc4', 'nc4/1980-03.nc4', 'nc4/1980-04.nc4', 'nc4/1980-05.nc4']
Data variables in first file:
['T2M', 'T2MDEW', 'Var_T2M', 'T2MWET']
Coordinates:
{'lon': (576,), 'time': (1,), 'lat': (361,)}
Confirmed var_name = T2M
Lat: 361, Lon: 576


Combining: 100%|██████████| 548/548 [00:29<00:00, 18.82it/s]


Combined data shape: (548, 361, 576)
Time index: 1980-01-01 00:30:00 -> 2025-08-01 00:30:00 (len=548)
y_all shape: (207936, 548)  (cells x time)
gg shape: (207936, 2)        (cells x [lon, lat])


# 限制範圍抽樣 in USA

## 25個格點

In [2]:
# ===== 只看美國本土範圍，並從中抽樣 25 個格點 =====
import numpy as np

# gg: (ncell, 2)；第 0 欄是 lon，第 1 欄是 lat
lon_all = gg[:, 0].astype(float)
lat_all = gg[:, 1].astype(float)

# 如果經度是 0~360，轉成 -180~180（如果本來就是 -180~180，這個轉換也不會壞掉）
lon_all_180 = ((lon_all + 180) % 360) - 180

# 美國本土 48 州的大致範圍（可以再微調）
lon_min, lon_max = -125, -66
lat_min, lat_max = 24, 50

# 建立遮罩：只保留在美國本土範圍內的格點
mask_us_mainland = (
    (lon_all_180 >= lon_min) & (lon_all_180 <= lon_max) &
    (lat_all      >= lat_min) & (lat_all      <= lat_max)
)

idx_us_mainland = np.where(mask_us_mainland)[0]
print(f"美國本土範圍內的格點數量: {len(idx_us_mainland)}")

if len(idx_us_mainland) == 0:
    raise ValueError("在設定的美國本土範圍內沒有格點，請調整 lon_min/max 或 lat_min/max。")

# 只保留美國本土子集合
y_us = y_all[idx_us_mainland, :]  # (n_us, ntot)
coords_us = np.column_stack([lon_all_180[idx_us_mainland], lat_all[idx_us_mainland]])  # (n_us, 2)

print(f"美國本土子集合 y_us shape: {y_us.shape}")
print(f"美國本土子集合 coords_us shape: {coords_us.shape}")

# 從美國本土格點中隨機抽樣 25 個（如果不足 25，就全部使用）
n_sample = min(25, len(idx_us_mainland))
np.random.seed(42)  # 為了可重現
sample_idx_in_us = np.random.choice(len(idx_us_mainland), size=n_sample, replace=False)

# 這是「在原始格點編號中的 index」
sample_idx_global = idx_us_mainland[sample_idx_in_us]

# 抽樣後的 time series 與座標
y_sample = y_all[sample_idx_global, :]  # (n_sample, ntot)
coords_sample = np.column_stack([lon_all_180[sample_idx_global], lat_all[sample_idx_global]])  # (n_sample, 2)

# 如果之後畫圖想分開拿 lon / lat：
lon_us_sample = coords_sample[:, 0]
lat_us_sample = coords_sample[:, 1]

print(f"抽樣 {n_sample} 個美國本土格點")
print("前 5 個 sample index (global):", sample_idx_global[:5])
print("前 5 個 sample 座標 (lon, lat):")
for i in range(min(5, n_sample)):
    print(f"  #{i}: lon={lon_us_sample[i]:.2f}, lat={lat_us_sample[i]:.2f}")


美國本土範圍內的格點數量: 5035
美國本土子集合 y_us shape: (5035, 548)
美國本土子集合 coords_us shape: (5035, 2)
抽樣 25 個美國本土格點
前 5 個 sample index (global): [153335 159677 145263 156825 138939]
前 5 個 sample 座標 (lon, lat):
  #0: lon=-105.62, lat=43.00
  #1: lon=-101.88, lat=48.50
  #2: lon=-110.62, lat=36.00
  #3: lon=-84.38, lat=46.00
  #4: lon=-103.12, lat=30.50


### DLinear

In [3]:
# ===== Final DLinear with your BEST setting (25 sampled US grid cells) =====
import numpy as np
import pandas as pd

from darts import TimeSeries
from darts.models import DLinearModel
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("\n===== FINAL DLinear (BEST setting) on 25 sampled US cells =====")

# ============================================================
# 0) BEST setting (你提供的最佳組合)
# ============================================================
INPUT_CHUNK_LENGTH  = 36
OUTPUT_CHUNK_LENGTH = 24
KERNEL_SIZE         = 25
N_EPOCHS            = 50

USE_COVARIATES       = False
USE_REVIN            = True

BATCH_SIZE           = 64
LR                   = 0.0003
WEIGHT_DECAY         = 0.0
CONST_INIT           = True

RANDOM_STATE         = 42

# Trainer 輸出壓到最低，避免爆版
PL_TRAINER_KWARGS = {
    "enable_progress_bar": False,
    "logger": False,
    "enable_checkpointing": False,
}

# ============================================================
# 1) 建 target DataFrame：25 個格點 × T 時間點（原始單位）
# ============================================================
SAMPLE_SIZE = n_sample  # 預期是 25
T = y_sample.shape[1]

ts_df = pd.DataFrame(
    y_sample.T,   # (T, SAMPLE_SIZE)
    index=time_index,
    columns=[f"cell_{i}" for i in range(SAMPLE_SIZE)]
).astype("float32")

print("Total time steps:", T)
print("ts_df shape:", ts_df.shape)

# ============================================================
# 2) month covariate（保留；但 USE_COVARIATES=False 時不會用到）
# ============================================================
month_values = time_index.month.astype("float32")
month_df = pd.DataFrame({"month": month_values}, index=time_index)
month_ts = TimeSeries.from_dataframe(month_df.astype("float32"))

# ============================================================
# 3) 切成 70 / 15 / 15（train / val / test）
# ============================================================
train_frac = 0.7
val_frac   = 0.15

cut_train = int(T * train_frac)
cut_val   = int(T * (train_frac + val_frac))

idx = ts_df.index
split_1_time = idx[cut_train]
split_2_time = idx[cut_val]

ts_all_raw = TimeSeries.from_dataframe(ts_df)
train_raw, tmp_raw = ts_all_raw.split_before(split_1_time)
val_raw, test_raw  = tmp_raw.split_before(split_2_time)

print(f"Train len (raw): {len(train_raw)}, Val len: {len(val_raw)}, Test len: {len(test_raw)}")

month_train, tmp_month = month_ts.split_before(split_1_time)
month_val, month_test  = tmp_month.split_before(split_2_time)

assert len(month_train) == len(train_raw)
assert len(month_val) == len(val_raw)
assert len(month_test) == len(test_raw)

# ============================================================
# 4) 用 train 統計量標準化
# ============================================================
train_df = ts_df.iloc[:cut_train]
mean_vec = train_df.mean(axis=0)
std_vec  = train_df.std(axis=0).replace(0.0, 1.0)

ts_df_scaled  = (ts_df - mean_vec) / std_vec
ts_all_scaled = TimeSeries.from_dataframe(ts_df_scaled)

train_ts, tmp_ts = ts_all_scaled.split_before(split_1_time)
val_ts, test_ts  = tmp_ts.split_before(split_2_time)

T_train, T_val, T_test = len(train_ts), len(val_ts), len(test_ts)
print("\n[Scaled lengths] T_train =", T_train, "T_val =", T_val, "T_test =", T_test)

vals_all      = ts_df.to_numpy(dtype=np.float32)
val_true_raw  = vals_all[cut_train:cut_val, :]
test_true_raw = vals_all[cut_val:, :]

def inverse_scale(x: np.ndarray) -> np.ndarray:
    return x * std_vec.values + mean_vec.values

def _ts_to_2d(ts: TimeSeries) -> np.ndarray:
    arr = np.asarray(ts.all_values(copy=False))
    if arr.ndim == 3:
        arr = arr[..., 0]  # (T, C)
    return arr

# 用於 TEST 訓練：train+val
train_val_ts    = train_ts.concatenate(val_ts)
month_train_val = month_train.concatenate(month_val)

# ============================================================
# 5) 建立模型（VAL）
# ============================================================
model_kwargs = dict(
    input_chunk_length=INPUT_CHUNK_LENGTH,
    output_chunk_length=OUTPUT_CHUNK_LENGTH,
    kernel_size=KERNEL_SIZE,
    n_epochs=N_EPOCHS,
    random_state=RANDOM_STATE,
    use_reversible_instance_norm=USE_REVIN,
    batch_size=BATCH_SIZE,
    optimizer_kwargs={"lr": LR, "weight_decay": WEIGHT_DECAY},
    const_init=CONST_INIT,
    pl_trainer_kwargs=PL_TRAINER_KWARGS,
    log_tensorboard=False,
    save_checkpoints=False,
)

model_val = DLinearModel(**model_kwargs)

fit_kwargs_val = {"series": train_ts, "verbose": False}
pred_kwargs_val = {"n": T_val, "verbose": False, "show_warnings": False}

if USE_COVARIATES:
    fit_kwargs_val["past_covariates"] = month_train
    fit_kwargs_val["future_covariates"] = month_train
    pred_kwargs_val["past_covariates"] = month_ts
    pred_kwargs_val["future_covariates"] = month_ts

model_val.fit(**fit_kwargs_val)
pred_val = model_val.predict(**pred_kwargs_val)

pred_val_scaled = _ts_to_2d(pred_val)
pred_val_raw = inverse_scale(pred_val_scaled)

# ============================================================
# 6) 建立模型（TEST）：train+val → predict test
# ============================================================
model_test = DLinearModel(**model_kwargs)

fit_kwargs_test = {"series": train_val_ts, "verbose": False}
pred_kwargs_test = {"n": T_test, "verbose": False, "show_warnings": False}

if USE_COVARIATES:
    fit_kwargs_test["past_covariates"] = month_train_val
    fit_kwargs_test["future_covariates"] = month_train_val
    pred_kwargs_test["past_covariates"] = month_ts
    pred_kwargs_test["future_covariates"] = month_ts

model_test.fit(**fit_kwargs_test)
pred_test = model_test.predict(**pred_kwargs_test)

pred_test_scaled = _ts_to_2d(pred_test)
pred_test_raw = inverse_scale(pred_test_scaled)

# ============================================================
# 7) 計算指標（VAL / TEST）
# ============================================================
y_val_true  = val_true_raw.reshape(-1, SAMPLE_SIZE)
y_val_pred  = pred_val_raw.reshape(-1, SAMPLE_SIZE)
y_test_true = test_true_raw.reshape(-1, SAMPLE_SIZE)
y_test_pred = pred_test_raw.reshape(-1, SAMPLE_SIZE)

mse_val  = mean_squared_error(y_val_true,  y_val_pred)
mae_val  = mean_absolute_error(y_val_true, y_val_pred)
rmse_val = float(np.sqrt(mse_val))
r2_val   = r2_score(y_val_true, y_val_pred)

mse_test  = mean_squared_error(y_test_true,  y_test_pred)
mae_test  = mean_absolute_error(y_test_true, y_test_pred)
rmse_test = float(np.sqrt(mse_test))
r2_test   = r2_score(y_test_true, y_test_pred)

metrics_table = pd.DataFrame({
    "Model": ["DLINEAR", "DLINEAR"],
    "Split": ["VAL", "TEST"],
    "RMSE": [rmse_val, rmse_test],
    "MSE":  [mse_val,  mse_test],
    "MAE":  [mae_val,  mae_test],
    "R2":   [r2_val,   r2_test],
})

print("\n===== FINAL DLINEAR RESULTS (BEST setting) =====")
print(metrics_table.to_string(index=False))


/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/fs/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore
/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs



===== FINAL DLinear (BEST setting) on 25 sampled US cells =====
Total time steps: 548
ts_df shape: (548, 25)
Train len (raw): 383, Val len: 82, Test len: 83

[Scaled lengths] T_train = 383 T_val = 82 T_test = 83


/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`Trainer.fit` stopped: `max_epochs=50` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`Trainer.fit` stopped: `max_epochs=50` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs



===== FINAL DLINEAR RESULTS (BEST setting) =====
  Model Split     RMSE      MSE      MAE       R2
DLINEAR   VAL 1.929747 3.723924 1.426790 0.871236
DLINEAR  TEST 1.546434 2.391459 1.124294 0.945872


### autoFRK

In [4]:
# =========================
# DLinear + autoFRK latent residual dynamics forecast
# =========================

import numpy as np
import torch
from autoFRK import AutoFRK

# ------------------------------------------------------------
# Coordinates (N, 2) in (lon, lat)
# ------------------------------------------------------------
# coords_sample: 25 個格點的經緯度
coords = coords_sample.astype(np.float64)

# autoFRK 需要 torch tensor 作為位置輸入
loc = torch.from_numpy(coords).to(dtype=torch.float64, device="cpu")

N = coords.shape[0]   # 空間格點數（例如 25）
H = T_test            # 未來預測時間長度（TEST 時段長度）

# ------------------------------------------------------------
# Helpers: 將 DLinear 的 scaled 預測轉回原始尺度
# ------------------------------------------------------------
mean_arr = mean_vec.values.astype(np.float64)  # 每個格點的 mean
std_arr  = std_vec.values.astype(np.float64)   # 每個格點的 std

def inverse_scale_2d(x_TN):
    """
    將 scaled 預測還原為原始單位。
    x_TN: (T, N) → 每個時間、每個格點
    """
    return x_TN * std_arr[None, :] + mean_arr[None, :]

def ts_to_TN(ts):
    """
    將 Darts TimeSeries 轉為 numpy (T, N)。
    """
    arr = np.asarray(ts.all_values(copy=False))
    if arr.ndim == 3:
        arr = arr[..., 0]  # (T, N)
    return arr.astype(np.float64, copy=False)

# ------------------------------------------------------------
# 1) 建立歷史 residual 資料 (train + val)
# ------------------------------------------------------------
# 真實歷史資料（raw）：(T_hist, N)
y_hist_raw_TN = ts_df.iloc[:cut_val, :].to_numpy(dtype=np.float64)
hist_index = ts_df.index[:cut_val]
T_hist = y_hist_raw_TN.shape[0]

# ------------------------------------------------------------
# 1-step historical forecast（scaled）
# ------------------------------------------------------------
# 用過去資料預測當下 → 真正 forecast error
pred_hist = model_test.historical_forecasts(
    series=train_val_ts,
    start=INPUT_CHUNK_LENGTH,   # 前 INPUT_CHUNK_LENGTH 無法預測
    forecast_horizon=1,         # 1-step ahead
    stride=1,
    retrain=False,
    last_points_only=True,
    verbose=False,
)

# Darts 版本相容處理
if isinstance(pred_hist, list):
    pred_hist_ts = pred_hist[0]
    for k in range(1, len(pred_hist)):
        pred_hist_ts = pred_hist_ts.append(pred_hist[k])
else:
    pred_hist_ts = pred_hist

# scaled 預測 (T_pred, N)
pred_hist_scaled_TN = ts_to_TN(pred_hist_ts)
pred_time_index = pred_hist_ts.time_index

# ------------------------------------------------------------
# 將預測對齊回完整歷史時間軸
# ------------------------------------------------------------
trend_hat_hist_raw_TN = np.full((T_hist, N), np.nan)

# 找到預測對應的時間位置
pos = hist_index.get_indexer(pred_time_index)
valid = pos >= 0
pred_hist_scaled_TN = pred_hist_scaled_TN[valid]
pos = pos[valid]

# 還原為 raw
pred_hist_raw_TN = inverse_scale_2d(pred_hist_scaled_TN)

# 填入對齊後的趨勢預測
trend_hat_hist_raw_TN[pos] = pred_hist_raw_TN

# ------------------------------------------------------------
# 計算 residual（DLinear forecast error）
# ------------------------------------------------------------
# r(t,s) = 真實值 - DLinear 預測
R_hist_raw_TN = y_hist_raw_TN - trend_hat_hist_raw_TN

# autoFRK 需要 (N, T)
R_hist_raw_NT = R_hist_raw_TN.T

print("Residual history shape:", R_hist_raw_NT.shape,
      "| NaN ratio:", float(np.isnan(R_hist_raw_NT).mean()))

# ------------------------------------------------------------
# 2) 用 autoFRK 學 residual 的空間結構
# ------------------------------------------------------------
afrk = AutoFRK(dtype=torch.float64, device="cpu")

# residual 資料轉 torch tensor
data_hist = torch.from_numpy(R_hist_raw_NT).to(dtype=torch.float64)

# autoFRK 分解：
# r(t,s) ≈ Φ(s)^T w(t)
result = afrk.forward(
    data=data_hist,
    loc=loc,
    method="EM",                 # EM 可處理 NaN
    maxit=50,
    tolerance=1e-6,
    n_neighbor=3,
    tps_method="spherical_fast", # 經緯度用球面距離
)

# latent 空間因子 w(t)：(K, T_hist)
w_hist = result["w"]
w_hist_np = w_hist.detach().cpu().numpy()
K, T_hist = w_hist_np.shape

# ------------------------------------------------------------
# 3) VAR(1) 建立 latent dynamics
# ------------------------------------------------------------
# w_{t+1} = A w_t
W0 = w_hist_np[:, :-1]
W1 = w_hist_np[:, 1:]
A = (W1 @ W0.T) @ np.linalg.pinv(W0 @ W0.T)

# ------------------------------------------------------------
# 4) 預測未來 latent 因子
# ------------------------------------------------------------
w_last = w_hist_np[:, -1]
w_fut = np.zeros((K, H))

wk = w_last
for h in range(H):
    wk = A @ wk
    w_fut[:, h] = wk

w_fut_t = torch.from_numpy(w_fut).to(dtype=torch.float64)

# ------------------------------------------------------------
# 5) 將 latent 因子轉回 residual 空間場
# ------------------------------------------------------------
obj_fut = dict(result)
obj_fut["w"] = w_fut_t

pred_res = afrk.predict(
    obj=obj_fut,
    newloc=loc,
    se_report=False,
    tps_method="spherical_fast",
)

# residual 預測：(N, H)
Rhat_NH = pred_res["pred.value"].detach().cpu().numpy()

# ------------------------------------------------------------
# 6) 合成最終預測：趨勢 + residual
# ------------------------------------------------------------
trend_test_raw = pred_test_raw.astype(np.float64)
Yhat_test = trend_test_raw + Rhat_NH.T

# ------------------------------------------------------------
# 7) 計算 TEST RMSE
# ------------------------------------------------------------
y_test_true = test_true_raw.reshape(H, N).astype(np.float64)
rmse_test_combo = float(np.sqrt(np.mean((y_test_true - Yhat_test) ** 2)))

print("\nDLinear + autoFRK forecast RMSE(test) =", rmse_test_combo)



GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
2026-03-01 23:20:29 - autoFRK.utils.logger - INFO: Successfully using device "cpu".
2026-03-01 23:20:29 - autoFRK.utils.logger - INFO: Calculate TPS with spherical_fast.


Residual history shape: (25, 465) | NaN ratio: 0.07741935483870968


2026-03-01 23:20:30 - autoFRK.utils.logger - INFO: Number of iteration: 1



DLinear + autoFRK forecast RMSE(test) = 1.540301094066429


### DLinear + auotFRK

In [5]:
# ============================================================
# Final DLinear (BEST setting) + autoFRK latent residual dynamics forecast
# 25 sampled US grid cells
# ============================================================

import numpy as np
import pandas as pd

import torch
from autoFRK import AutoFRK

from darts import TimeSeries
from darts.models import DLinearModel
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


print("\n===== FINAL DLinear (BEST setting) + autoFRK (25 sampled US cells) =====")

# ============================================================
# 0) BEST setting (你提供的最佳組合)
# ============================================================
INPUT_CHUNK_LENGTH  = 36
OUTPUT_CHUNK_LENGTH = 24
KERNEL_SIZE         = 25
N_EPOCHS            = 50

USE_COVARIATES       = False
USE_REVIN            = True

BATCH_SIZE           = 64
LR                   = 0.0003
WEIGHT_DECAY         = 0.0
CONST_INIT           = True

RANDOM_STATE         = 42

# Trainer 輸出壓到最低，避免爆版
PL_TRAINER_KWARGS = {
    "enable_progress_bar": False,
    "logger": False,
    "enable_checkpointing": False,
}

# ============================================================
# 1) 建 target DataFrame：25 個格點 × T 時間點（原始單位）
#    你需要先在外部準備好：
#    - y_sample: shape = (SAMPLE_SIZE, T) 或可轉置成 (T, SAMPLE_SIZE)
#    - time_index: pandas.DatetimeIndex, length = T
#    - n_sample: int, 預期 25
# ============================================================
SAMPLE_SIZE = n_sample
T = y_sample.shape[1]

ts_df = pd.DataFrame(
    y_sample.T,  # (T, SAMPLE_SIZE)
    index=time_index,
    columns=[f"cell_{i}" for i in range(SAMPLE_SIZE)]
).astype("float32")

print("Total time steps:", T)
print("ts_df shape:", ts_df.shape)

# ============================================================
# 2) month covariate（保留；USE_COVARIATES=False 時不會用到）
# ============================================================
month_values = time_index.month.astype("float32")
month_df = pd.DataFrame({"month": month_values}, index=time_index)
month_ts = TimeSeries.from_dataframe(month_df.astype("float32"))

# ============================================================
# 3) 切成 70 / 15 / 15（train / val / test）
# ============================================================
train_frac = 0.7
val_frac   = 0.15

cut_train = int(T * train_frac)
cut_val   = int(T * (train_frac + val_frac))

idx = ts_df.index
split_1_time = idx[cut_train]
split_2_time = idx[cut_val]

ts_all_raw = TimeSeries.from_dataframe(ts_df)
train_raw, tmp_raw = ts_all_raw.split_before(split_1_time)
val_raw, test_raw  = tmp_raw.split_before(split_2_time)

print(f"Train len (raw): {len(train_raw)}, Val len: {len(val_raw)}, Test len: {len(test_raw)}")

month_train, tmp_month = month_ts.split_before(split_1_time)
month_val, month_test  = tmp_month.split_before(split_2_time)

assert len(month_train) == len(train_raw)
assert len(month_val) == len(val_raw)
assert len(month_test) == len(test_raw)

# ============================================================
# 4) 用 train 統計量標準化（只用 train 的 mean/std）
# ============================================================
train_df = ts_df.iloc[:cut_train]
mean_vec = train_df.mean(axis=0)
std_vec  = train_df.std(axis=0).replace(0.0, 1.0)

ts_df_scaled  = (ts_df - mean_vec) / std_vec
ts_all_scaled = TimeSeries.from_dataframe(ts_df_scaled)

train_ts, tmp_ts = ts_all_scaled.split_before(split_1_time)
val_ts, test_ts  = tmp_ts.split_before(split_2_time)

T_train, T_val, T_test = len(train_ts), len(val_ts), len(test_ts)
print("\n[Scaled lengths] T_train =", T_train, "T_val =", T_val, "T_test =", T_test)

vals_all      = ts_df.to_numpy(dtype=np.float32)
val_true_raw  = vals_all[cut_train:cut_val, :]
test_true_raw = vals_all[cut_val:, :]

def inverse_scale_1d_or_2d(x: np.ndarray) -> np.ndarray:
    """
    將 scaled 預測還原為原始單位。
    x shape:
      - (N,) 或 (T, N)
    """
    return x * std_vec.values + mean_vec.values

def ts_to_TN(ts: TimeSeries) -> np.ndarray:
    """
    將 Darts TimeSeries 轉為 numpy (T, N)。
    """
    arr = np.asarray(ts.all_values(copy=False))
    if arr.ndim == 3:
        arr = arr[..., 0]  # (T, N)
    return arr

# 用於 TEST 訓練：train+val
train_val_ts    = train_ts.concatenate(val_ts)
month_train_val = month_train.concatenate(month_val)

# ============================================================
# 5) 建立模型（VAL）：只用 train fit，然後預測整段 val
# ============================================================
model_kwargs = dict(
    input_chunk_length=INPUT_CHUNK_LENGTH,
    output_chunk_length=OUTPUT_CHUNK_LENGTH,
    kernel_size=KERNEL_SIZE,
    n_epochs=N_EPOCHS,
    random_state=RANDOM_STATE,
    use_reversible_instance_norm=USE_REVIN,
    batch_size=BATCH_SIZE,
    optimizer_kwargs={"lr": LR, "weight_decay": WEIGHT_DECAY},
    const_init=CONST_INIT,
    pl_trainer_kwargs=PL_TRAINER_KWARGS,
    log_tensorboard=False,
    save_checkpoints=False,
)

model_val = DLinearModel(**model_kwargs)

fit_kwargs_val = {"series": train_ts, "verbose": False}
pred_kwargs_val = {"n": T_val, "verbose": False, "show_warnings": False}

if USE_COVARIATES:
    fit_kwargs_val["past_covariates"] = month_train
    fit_kwargs_val["future_covariates"] = month_train
    pred_kwargs_val["past_covariates"] = month_ts
    pred_kwargs_val["future_covariates"] = month_ts

model_val.fit(**fit_kwargs_val)
pred_val = model_val.predict(**pred_kwargs_val)

pred_val_scaled_TN = ts_to_TN(pred_val).astype(np.float64)
pred_val_raw_TN = inverse_scale_1d_or_2d(pred_val_scaled_TN)

# ============================================================
# 6) 建立模型（TEST）：train+val fit，然後預測整段 test
# ============================================================
model_test = DLinearModel(**model_kwargs)

fit_kwargs_test = {"series": train_val_ts, "verbose": False}
pred_kwargs_test = {"n": T_test, "verbose": False, "show_warnings": False}

if USE_COVARIATES:
    fit_kwargs_test["past_covariates"] = month_train_val
    fit_kwargs_test["future_covariates"] = month_train_val
    pred_kwargs_test["past_covariates"] = month_ts
    pred_kwargs_test["future_covariates"] = month_ts

model_test.fit(**fit_kwargs_test)
pred_test = model_test.predict(**pred_kwargs_test)

pred_test_scaled_TN = ts_to_TN(pred_test).astype(np.float64)
pred_test_raw_TN = inverse_scale_1d_or_2d(pred_test_scaled_TN)

# ============================================================
# 7) DLinear 指標（VAL / TEST）
# ============================================================
y_val_true  = val_true_raw.reshape(-1, SAMPLE_SIZE)
y_val_pred  = pred_val_raw_TN.reshape(-1, SAMPLE_SIZE)
y_test_true = test_true_raw.reshape(-1, SAMPLE_SIZE)
y_test_pred = pred_test_raw_TN.reshape(-1, SAMPLE_SIZE)

mse_val  = mean_squared_error(y_val_true,  y_val_pred)
mae_val  = mean_absolute_error(y_val_true, y_val_pred)
rmse_val = float(np.sqrt(mse_val))
r2_val   = r2_score(y_val_true, y_val_pred)

mse_test  = mean_squared_error(y_test_true,  y_test_pred)
mae_test  = mean_absolute_error(y_test_true, y_test_pred)
rmse_test = float(np.sqrt(mse_test))
r2_test   = r2_score(y_test_true, y_test_pred)

metrics_table = pd.DataFrame({
    "Model": ["DLINEAR", "DLINEAR"],
    "Split": ["VAL", "TEST"],
    "RMSE": [rmse_val, rmse_test],
    "MSE":  [mse_val,  mse_test],
    "MAE":  [mae_val,  mae_test],
    "R2":   [r2_val,   r2_test],
})

print("\n===== FINAL DLINEAR RESULTS (BEST setting) =====")
print(metrics_table.to_string(index=False))

# ============================================================
# 8) autoFRK：用 DLinear 的 1-step forecast error 建 residual 歷史資料
#    然後在 latent 空間做 VAR(1) 推進，再轉回 residual 場做修正
#    你需要先在外部準備好：
#    - coords_sample: shape = (N, 2), (lon, lat)
# ============================================================

# 8.1 位置資料
coords = coords_sample.astype(np.float64)
loc = torch.from_numpy(coords).to(dtype=torch.float64, device="cpu")

N = coords.shape[0]
H = T_test  # 未來預測時間長度（TEST 時段長度）

# 8.2 取 train+val 的真實歷史資料（raw）
y_hist_raw_TN = ts_df.iloc[:cut_val, :].to_numpy(dtype=np.float64)  # (T_hist, N)
hist_index = ts_df.index[:cut_val]
T_hist = y_hist_raw_TN.shape[0]

# 8.3 用 train+val 的模型，在 (train+val) 上做 1-step historical forecast
#     目的：拿到「真正的 forecast error」，避免把 in-sample 擬合殘差當作 forecast residual
pred_hist = model_test.historical_forecasts(
    series=train_val_ts,
    start=INPUT_CHUNK_LENGTH,   # 前 INPUT_CHUNK_LENGTH 無法預測
    forecast_horizon=1,         # 1-step ahead
    stride=1,
    retrain=False,
    last_points_only=True,
    verbose=False,
)

# Darts 版本相容處理：可能回傳 list[TimeSeries] 或單一 TimeSeries
if isinstance(pred_hist, list):
    pred_hist_ts = pred_hist[0]
    for k in range(1, len(pred_hist)):
        pred_hist_ts = pred_hist_ts.append(pred_hist[k])
else:
    pred_hist_ts = pred_hist

pred_hist_scaled_TN = ts_to_TN(pred_hist_ts).astype(np.float64)  # (T_pred, N)
pred_time_index = pred_hist_ts.time_index

# 8.4 將 1-step 預測對齊回完整歷史時間軸
#     沒有預測的時間點會是 NaN（例如起始 INPUT_CHUNK_LENGTH 之前）
trend_hat_hist_raw_TN = np.full((T_hist, N), np.nan, dtype=np.float64)

pos = hist_index.get_indexer(pred_time_index)  # 對齊位置
valid = pos >= 0

pred_hist_scaled_TN = pred_hist_scaled_TN[valid]
pos = pos[valid]

pred_hist_raw_TN = inverse_scale_1d_or_2d(pred_hist_scaled_TN)  # 還原 raw
trend_hat_hist_raw_TN[pos] = pred_hist_raw_TN

# 8.5 residual：r(t,s) = y(t,s) - trend_hat(t,s)
R_hist_raw_TN = y_hist_raw_TN - trend_hat_hist_raw_TN  # (T_hist, N)

# autoFRK 期望 (N, T)
R_hist_raw_NT = R_hist_raw_TN.T  # (N, T_hist)

print("\n===== Residual history (for autoFRK) =====")
print("Residual history shape (N, T_hist):", R_hist_raw_NT.shape)
print("NaN ratio:", float(np.isnan(R_hist_raw_NT).mean()))

# ============================================================
# 9) autoFRK：學 residual 的空間結構（EM 可處理 NaN）
# ============================================================
afrk = AutoFRK(dtype=torch.float64, device="cpu")
data_hist = torch.from_numpy(R_hist_raw_NT).to(dtype=torch.float64)

# 分解：r(t,s) ≈ Φ(s)^T w(t)
result = afrk.forward(
    data=data_hist,
    loc=loc,
    method="EM",
    maxit=50,
    tolerance=1e-6,
    n_neighbor=3,
    tps_method="spherical_fast",
)

w_hist = result["w"]  # (K, T_hist)
w_hist_np = w_hist.detach().cpu().numpy()
K, T_hist_check = w_hist_np.shape

if T_hist_check != T_hist:
    print("Warning: w_hist length mismatch:", T_hist_check, "vs", T_hist)

print("\n===== autoFRK latent factors =====")
print("K (latent dim):", K)
print("T_hist (latent time):", T_hist_check)

# ============================================================
# 10) latent dynamics：VAR(1) 在 w(t) 上做時間推進
#     w_{t+1} = A w_t
# ============================================================
W0 = w_hist_np[:, :-1]
W1 = w_hist_np[:, 1:]

A = (W1 @ W0.T) @ np.linalg.pinv(W0 @ W0.T)

# ============================================================
# 11) 預測未來 latent 因子（長度 = H = T_test）
# ============================================================
w_last = w_hist_np[:, -1]
w_fut = np.zeros((K, H), dtype=np.float64)

wk = w_last
for h in range(H):
    wk = A @ wk
    w_fut[:, h] = wk

w_fut_t = torch.from_numpy(w_fut).to(dtype=torch.float64)

# ============================================================
# 12) 將未來 latent 因子轉回 residual 空間場：rhat(s, t)
# ============================================================
obj_fut = dict(result)
obj_fut["w"] = w_fut_t

pred_res = afrk.predict(
    obj=obj_fut,
    newloc=loc,
    se_report=False,
    tps_method="spherical_fast",
)

# residual 預測：(N, H)
Rhat_NH = pred_res["pred.value"].detach().cpu().numpy()

# ============================================================
# 13) 合成最終 TEST 預測：trend + residual
#     trend_test_raw_TN: (H, N)
#     Rhat_NH.T:         (H, N)
# ============================================================
trend_test_raw_TN = pred_test_raw_TN.astype(np.float64)
Yhat_test_TN = trend_test_raw_TN + Rhat_NH.T  # (H, N)

# ============================================================
# 14) 計算 DLinear + autoFRK 的 TEST RMSE
# ============================================================
y_test_true_TN = test_true_raw.reshape(H, N).astype(np.float64)
rmse_test_combo = float(np.sqrt(np.mean((y_test_true_TN - Yhat_test_TN) ** 2)))

print("\n===== DLinear + autoFRK RESULTS =====")
print("DLinear RMSE(test)           =", rmse_test)
print("DLinear + autoFRK RMSE(test) =", rmse_test_combo)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs



===== FINAL DLinear (BEST setting) + autoFRK (25 sampled US cells) =====
Total time steps: 548
ts_df shape: (548, 25)
Train len (raw): 383, Val len: 82, Test len: 83

[Scaled lengths] T_train = 383 T_val = 82 T_test = 83


/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`Trainer.fit` stopped: `max_epochs=50` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/jundian/miniconda3/envs/geospatial-neural-adapter/lib/python3.10/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`Trainer.fit` stopped: `max_epochs=50` reached.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: False, used: False
TPU


===== FINAL DLINEAR RESULTS (BEST setting) =====
  Model Split     RMSE      MSE      MAE       R2
DLINEAR   VAL 1.929747 3.723924 1.426790 0.871236
DLINEAR  TEST 1.546435 2.391460 1.124294 0.945872

===== Residual history (for autoFRK) =====
Residual history shape (N, T_hist): (25, 465)
NaN ratio: 0.07741935483870968


2026-03-01 23:20:37 - autoFRK.utils.logger - INFO: Number of iteration: 1



===== autoFRK latent factors =====
K (latent dim): 21
T_hist (latent time): 465

===== DLinear + autoFRK RESULTS =====
DLinear RMSE(test)           = 1.5464346004348493
DLinear + autoFRK RMSE(test) = 1.540301286479433
